# TRIBE v2 — Cognitive Load Demo on Merge Conflicts

Demonstração mínima de uso do TRIBE v2 (Meta, 2026) para estimar carga cognitiva induzida por blocos de conflito de merge.

**Como rodar no Colab Free:**
1. Upload da pasta `tribe/` inteira (via painel lateral → Files).
2. Runtime → Change runtime type → GPU (T4).
3. Executar células **em ordem**.
4. **OBRIGATÓRIO:** após a célula de install, reiniciar o runtime (Runtime → Restart session) antes de seguir.

## 1. Install — clonar repo + alinhar torch/torchaudio

O pyproject do TRIBE pina `torch<2.7`, o que rebaixa o torch padrão do Colab mas não rebaixa o `torchaudio`. Sem alinhar manualmente, o import falha silenciosamente.

In [ ]:
# Clone com nome diferente para evitar shadow do pacote
!rm -rf /content/tribev2_repo
!git clone -q https://github.com/facebookresearch/tribev2.git /content/tribev2_repo

In [ ]:
%cd /content/tribev2_repo
!pip install -q -e ".[plotting]"
%cd /content

In [ ]:
# Alinha torchaudio com o torch que o tribev2 fixou (>=2.5.1,<2.7).
# --no-deps impede pip de voltar o torch pra 2.10.
!pip install -q "torchaudio>=2.5.1,<2.7" --no-deps --force-reinstall

## 2. REINICIAR O RUNTIME AGORA

**Runtime → Restart session** (ou Ctrl+M .). Depois pular a célula de install e continuar daqui.

Sem reiniciar, o torch antigo continua carregado em memória e os imports vão falhar.

## 3. Verificar GPU

In [ ]:
import torch
print("torch:", torch.__version__)
import torchaudio
print("torchaudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 4. Import diagnóstico

Se algo quebrar no import, esta célula mostra a stack trace real (em vez da mensagem enganosa `unknown location`).

In [ ]:
import sys, traceback

# Garante que /content (que contém a pasta tribev2_repo) não sombreie o pacote instalado
sys.path = [p for p in sys.path if p not in ("", "/content")]
for mod in list(sys.modules):
    if mod.startswith("tribev2"):
        del sys.modules[mod]

try:
    import tribev2
    print("tribev2.__file__:", tribev2.__file__)
    from tribev2 import TribeModel
    print("TribeModel imported OK")
except Exception:
    print("=== REAL ERROR BELOW ===")
    traceback.print_exc()

## 5. Carregar o modelo

Primeira execução baixa os pesos (alguns GB). Subsequentes usam cache local da sessão.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tribev2 import TribeModel

model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded.")

## 6. Listar conflitos de teste

In [ ]:
# Ajusta o caminho se a pasta conflicts/ estiver em outro lugar no Colab.
conflict_dir = Path("/content/conflicts")
if not conflict_dir.exists():
    # fallback: pasta no diretório atual
    conflict_dir = Path("conflicts")

conflicts = sorted(conflict_dir.glob("*.txt"))
print(f"Found {len(conflicts)} conflict file(s) in {conflict_dir}:")
for c in conflicts:
    print(f"  - {c.name} ({c.stat().st_size} bytes)")

## 7. Predição de carga cognitiva por conflito

Para cada conflito, o TRIBE converte texto em fala internamente, processa pelos encoders (LLaMA 3.2, Wav2Vec-BERT) e prediz a resposta fMRI na malha cortical fsaverage5 (~20k vértices).

**Proxy de carga (versão crua):**
- `mean_load` = magnitude média da ativação predita.
- `peak_load` = maior média de ativação em um único timestep.

In [ ]:
results = []
for path in conflicts:
    print(f"\nProcessing {path.name}...")
    df_events = model.get_events_dataframe(text_path=str(path))
    preds, segments = model.predict(events=df_events)
    mean_load = float(np.mean(np.abs(preds)))
    peak_load = float(np.max(np.mean(np.abs(preds), axis=1)))
    results.append({
        "conflict": path.stem,
        "timesteps": int(preds.shape[0]),
        "vertices": int(preds.shape[1]),
        "mean_load": mean_load,
        "peak_load": peak_load,
    })
    print(f"  shape: {preds.shape}")
    print(f"  mean_load: {mean_load:.4f}")
    print(f"  peak_load: {peak_load:.4f}")

## 8. Comparar resultados

In [ ]:
df_results = pd.DataFrame(results).sort_values("mean_load", ascending=False).reset_index(drop=True)
df_results

In [ ]:
df_results.to_csv("tribe_results.csv", index=False)
print("Saved to tribe_results.csv — faça download via painel lateral.")

## 9. Interpretação

**Se a hipótese for confirmada** (`scenario_38` > `01-method-conflict`): TRIBE diferencia dificuldade neurocognitiva entre tipos de conflito.

**Se inconclusivo ou invertido**: investigar:
- Normalizar carga por número de timesteps (scenario_38 é maior).
- Agregação global mascara sinais específicos — testar ROIs frontoparietais (Yeo-7).
- Conversão texto→fala pode descaracterizar código — considerar input visual.